# SDK + BYOT + Custom MCP

This notebook demonstrates the full Sec-Gemini stack: using the Python SDK alongside
BYOT to give the cloud agent access to baseline local tools **and** a custom FastMCP
server you define inline.

## Setup

Install the SDK and FastMCP.

In [ ]:
!pip install -q sec-gemini fastmcp

## Authentication

In [ ]:
import os

API_KEY = os.environ.get("SEC_GEMINI_API_KEY")

if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        API_KEY = userdata.get("SEC_GEMINI_API_KEY")
    except (ImportError, Exception):
        pass

if not API_KEY:
    API_KEY = "YOUR_API_KEY_HERE"

assert API_KEY and API_KEY != "YOUR_API_KEY_HERE", "Please set your API key"

## Define a Custom MCP

Create a FastMCP server with custom security tools. In a real workflow, this could
query your internal threat intelligence database, run YARA rules, or call internal APIs.

In [ ]:
from fastmcp import FastMCP

custom_mcp = FastMCP("custom-security-tools")


@custom_mcp.tool()
def lookup_hash(sha256: str) -> str:
    """Look up a file hash in the local threat intelligence database."""
    # Simulated lookup -- replace with real logic
    known_hashes = {
        "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855": "Known clean: empty file",
        "a" * 64: "MALICIOUS: Emotet dropper variant (TLP:RED)",
    }
    result = known_hashes.get(
        sha256.lower(), f"Hash {sha256[:16]}... not found in local DB"
    )
    return result


@custom_mcp.tool()
def check_ip_reputation(ip_address: str) -> str:
    """Check an IP address against the local reputation list."""
    # Simulated check -- replace with real logic
    blocklist = {"192.168.1.100": "Internal scanner", "10.0.0.1": "Gateway"}
    if ip_address in blocklist:
        return f"{ip_address}: {blocklist[ip_address]}"
    return f"{ip_address}: No reputation data in local DB"


print(
    f"Custom MCP defined with tools: {[t.name for t in custom_mcp.local_provider.tools.values()]}"  # type: ignore[attr-defined]
)

## Start BYOT with Baseline + Custom Tools

The `ByotService` connects to the BYOT Hub and registers both the baseline local tools
and your custom MCP.

In [ ]:
from sec_gemini.byot.service import ByotService
from sec_gemini.tools import create_baseline_mcp

# Create baseline tools
baseline_mcp, _tracker = create_baseline_mcp()

# Start BYOT with both tool sets
byot = ByotService(api_key=API_KEY, name="notebook-demo")
await byot.start(tools=[baseline_mcp, custom_mcp])

status = byot.status()
print(f"BYOT state: {status.state}")
print(f"Tools registered: {status.tool_count}")
for tool in status.tools:
    print(f"  - {tool.name}")

## Use the SDK with BYOT Active

Now the agent can call your custom tools alongside baseline and cloud tools.

In [ ]:
from sec_gemini import SecGemini

client = SecGemini(api_key=API_KEY)
await client.start()

session = await client.sessions.create()
print(f"Session: {session.id}")

# Ask the agent to use our custom tool
await session.prompt(
    "Look up this hash in the local threat database: "
    "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
)

async for msg in session.messages.stream():
    msg_type = msg.get("message_type", "")
    content = msg.get("content", "")
    title = msg.get("title", "")

    if msg_type == "MESSAGE_TYPE_RESPONSE":
        print(f"\n--- Agent Response ---\n{content}")
    elif msg_type == "MESSAGE_TYPE_TOOL_CALL":
        print(f"  [Tool Call] {title}")
    elif msg_type == "MESSAGE_TYPE_TOOL_RESULT":
        print(f"  [Tool Result] {content[:200]}")
    elif msg_type == "MESSAGE_TYPE_AGENT_IS_DONE":
        print("\n--- Done ---")

## Cleanup

Stop BYOT and close the SDK client.

In [ ]:
await session.delete()
await client.close()
await byot.stop()
print("BYOT stopped, client closed, session deleted.")